# 02 — BCRD CPI | Gold Inflation KPIs

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Calculate inflation KPIs **INF-001 to INF-006** from the Silver CPI tables into one model for Databricks SQL and Power BI, and reconcile every calculated value with the variation BCRD publishes.

## Target
`gold.inflation_metrics` — grain **Month × CPI series**, where a series is:

| series_type | series_name | Source |
|---|---|---|
| `headline` | National CPI | `silver.bcrd_cpi_national_monthly` |
| `core` | Core CPI | `silver.bcrd_cpi_core_monthly` |
| `expenditure_group` | 12 groups | `silver.bcrd_cpi_groups_monthly` |
| `region` | Ozama, Norte o Cibao, Este, Sur | `silver.bcrd_cpi_regions_monthly` |

## KPI mapping
| KPI | Column | Formula |
|---|---|---|
| INF-001 CPI Index | `index_value` | Published index |
| INF-002 Monthly Inflation | `mom_inflation_pct` | (CPI_t / CPI_t-1 − 1) × 100 |
| INF-003 YoY Inflation | `yoy_inflation_pct` | (CPI_t / CPI_t-12 − 1) × 100 |
| INF-004 YTD Inflation | `ytd_inflation_pct` | (CPI_t / CPI_Dec_prev_year − 1) × 100 |
| INF-005 Core Inflation | rows with `series_type = 'core'` | Same formulas on the official core index |
| INF-006 Inflation by Expenditure Group | rows with `series_type = 'expenditure_group'` | Same formulas on each group index |

Calculated values are reconciled against the BCRD published variations (`published_*_pct`). Windows without enough history are NULL with an `is_full_window_*` flag, the same rule as decision D4 for FX.

**Status:** Development — not production-ready.

In [0]:
# ============================================================
# Step 1 — Configuration
# ============================================================
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
S = f"{CATALOG}.silver"
TARGET_TABLE = f"{CATALOG}.gold.inflation_metrics"
RULE_RESULTS_TABLE = f"{CATALOG}.quality.dq_rule_results"
TOLERANCE_PP = 0.05   # calculated vs published variation, percentage points

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)
print(f"Run ID: {RUN_ID}")

In [0]:
# ============================================================
# Step 2 — Stack all CPI series into one long DataFrame
# ============================================================
null_d = F.lit(None).cast("decimal(12,4)")

headline = spark.table(f"{S}.bcrd_cpi_national_monthly").select(
    "reference_month", F.lit("headline").alias("series_type"), F.lit("National CPI").alias("series_name"),
    F.col("cpi_index").alias("index_value"),
    F.col("mom_pct").alias("published_mom_pct"), F.col("ytd_pct").alias("published_ytd_pct"),
    F.col("yoy_pct").alias("published_yoy_pct"), "base_period")

core = (spark.table(f"{S}.bcrd_cpi_core_monthly").filter("observation_type = 'monthly'").select(
    "reference_month", F.lit("core").alias("series_type"), F.lit("Core CPI").alias("series_name"),
    F.col("core_cpi_index").alias("index_value"),
    F.col("mom_pct").alias("published_mom_pct"), F.col("ytd_pct").alias("published_ytd_pct"),
    F.col("yoy_pct").alias("published_yoy_pct"), "base_period"))

groups = spark.table(f"{S}.bcrd_cpi_groups_monthly").select(
    "reference_month", F.lit("expenditure_group").alias("series_type"), F.col("expenditure_group").alias("series_name"),
    F.col("group_index").alias("index_value"),
    F.col("mom_pct").alias("published_mom_pct"), null_d.alias("published_ytd_pct"), null_d.alias("published_yoy_pct"), "base_period")

regions = spark.table(f"{S}.bcrd_cpi_regions_monthly").select(
    "reference_month", F.lit("region").alias("series_type"), F.col("region").alias("series_name"),
    F.col("cpi_index").alias("index_value"),
    F.col("mom_pct").alias("published_mom_pct"), null_d.alias("published_ytd_pct"), null_d.alias("published_yoy_pct"), "base_period")

series = headline.unionByName(core).unionByName(groups).unionByName(regions)
series = series.withColumn("idx", F.col("index_value").cast("double"))
print(f"Series rows: {series.count():,}")

In [0]:
# ============================================================
# Step 3 — Calculate INF-001 to INF-006
# ============================================================
w = Window.partitionBy("series_type", "series_name").orderBy("reference_month")

# December index of each year -> base for next year's YTD (INF-004)
december = (series.filter(F.month("reference_month") == 12)
            .select("series_type", "series_name",
                    (F.year("reference_month") + 1).alias("_year"), F.col("idx").alias("_dec_prev_idx")))

gold = (
    series
    .withColumn("_year", F.year("reference_month"))
    .withColumn("_prev_idx", F.lag("idx", 1).over(w))
    .withColumn("_prev_month", F.lag("reference_month", 1).over(w))
    .withColumn("_yago_idx", F.lag("idx", 12).over(w))
    .withColumn("_yago_month", F.lag("reference_month", 12).over(w))
    .join(december, ["series_type", "series_name", "_year"], "left")
    .withColumn("is_full_window_mom", F.coalesce(F.months_between("reference_month", "_prev_month") == 1, F.lit(False)))
    .withColumn("is_full_window_yoy", F.coalesce(F.months_between("reference_month", "_yago_month") == 12, F.lit(False)))
    .withColumn("is_full_window_ytd", F.col("_dec_prev_idx").isNotNull())
    .withColumn("mom_inflation_pct", F.when(F.col("is_full_window_mom"), (F.col("idx") / F.col("_prev_idx") - 1) * 100))
    .withColumn("yoy_inflation_pct", F.when(F.col("is_full_window_yoy"), (F.col("idx") / F.col("_yago_idx") - 1) * 100))
    .withColumn("ytd_inflation_pct", F.when(F.col("is_full_window_ytd"), (F.col("idx") / F.col("_dec_prev_idx") - 1) * 100))
)

def d4(c):
    return F.round(c, 4).cast("decimal(12,4)")

gold_out = gold.select(
    "reference_month", "series_type", "series_name",
    F.col("index_value").cast("decimal(12,4)").alias("index_value"),
    d4(F.col("mom_inflation_pct")).alias("mom_inflation_pct"), "is_full_window_mom",
    d4(F.col("yoy_inflation_pct")).alias("yoy_inflation_pct"), "is_full_window_yoy",
    d4(F.col("ytd_inflation_pct")).alias("ytd_inflation_pct"), "is_full_window_ytd",
    d4(F.col("published_mom_pct")).alias("published_mom_pct"),
    d4(F.col("published_yoy_pct")).alias("published_yoy_pct"),
    d4(F.col("published_ytd_pct")).alias("published_ytd_pct"),
    F.year("reference_month").alias("year"), F.month("reference_month").alias("month"),
    "base_period",
    F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"),
)

In [0]:
# ============================================================
# Step 4 — Validation and reconciliation with BCRD published values
# ============================================================
checks = []
def add(name, rtype, tested, failed):
    status = "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN")
    checks.append((RUN_ID, TARGET_TABLE, name, rtype, int(tested), int(failed), status, RUN_TS))

n = gold_out.count()
add("row_count_matches_silver", "enforced", n, abs(n - series.count()))
add("unique_month_series", "enforced", n, n - gold_out.select("reference_month", "series_type", "series_name").distinct().count())
for k in ["mom", "yoy", "ytd"]:
    add(f"{k}_null_iff_incomplete", "enforced", n,
        gold_out.filter(F.col(f"{k}_inflation_pct").isNull() == F.col(f"is_full_window_{k}")).count())

mismatch_examples = {}
for k in ["mom", "yoy", "ytd"]:
    comp = gold_out.filter(F.col(f"{k}_inflation_pct").isNotNull() & F.col(f"published_{k}_pct").isNotNull())
    bad = comp.filter(F.abs(F.col(f"{k}_inflation_pct") - F.col(f"published_{k}_pct")) > TOLERANCE_PP)
    add(f"{k}_reconciles_with_bcrd_published", "observational", comp.count(), bad.count())
    if bad.count():
        mismatch_examples[k] = [
            {c: str(r[c]) for c in ["reference_month", "series_type", "series_name", f"{k}_inflation_pct", f"published_{k}_pct"]}
            for r in bad.orderBy("reference_month").limit(5).collect()]

for c in checks:
    print(f"  {c[6]:<4} {c[2]:<36} ({c[3]}) tested={c[4]:,} failed={c[5]:,}")
if mismatch_examples:
    print(json.dumps(mismatch_examples, indent=1))

assert not [c for c in checks if c[6] == "FAIL"], "Enforced checks failed; nothing written"

In [0]:
# ============================================================
# Step 5 — Write Gold, document, log results
# ============================================================
(gold_out.orderBy("series_type", "series_name", "reference_month")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE))

spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Gold: inflation KPIs INF-001 to INF-006. Grain: month x CPI series (headline, core, 12 expenditure groups, 4 regions). Calculated values reconciled with BCRD published variations.'")
for col, comment in {
    "index_value": "INF-001: published CPI index (base Oct 2019 - Sep 2020 = 100)",
    "mom_inflation_pct": "INF-002: % change vs previous month",
    "yoy_inflation_pct": "INF-003: % change vs same month previous year; NULL until 12 months of history",
    "ytd_inflation_pct": "INF-004: % change vs December of previous year",
    "series_type": "headline | core (INF-005) | expenditure_group (INF-006) | region",
    "published_mom_pct": "BCRD published monthly variation, reference for reconciliation",
}.items():
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN {col} COMMENT '{comment}'")

spark.createDataFrame(checks,
    "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp",
).write.mode("append").saveAsTable(RULE_RESULTS_TABLE)

In [0]:
# ============================================================
# Step 6 — Business snapshot
# ============================================================
last = spark.table(TARGET_TABLE).agg(F.max("reference_month")).first()[0]
snap = (spark.table(TARGET_TABLE).filter(F.col("reference_month") == last)
        .select("series_type", "series_name", "index_value", "mom_inflation_pct", "yoy_inflation_pct", "ytd_inflation_pct")
        .orderBy("series_type", F.col("yoy_inflation_pct").desc()))
display(snap)

dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "rows": n,
    "latest_month": str(last),
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks},
    "mismatch_examples": mismatch_examples,
    "snapshot": [{k: str(v) for k, v in r.asDict().items()} for r in snap.collect()],
}, ensure_ascii=False))